# 🚀 Pipeline LEO Enhanced - Generador de Contenidos

Este notebook implementa 3 métodos para generar contenido de calidad:

1. **Por Tópico**: Define un tema amplio y LEO sugerirá una keyword óptima
2. **Por Keyword**: Usa directamente una keyword específica
3. **Por Newsletter/URL**: Analiza contenido de una newsletter para generar artículos

---

## 📦 Configuración Inicial

In [ ]:
# Importar librerías
import sys
import os
from dotenv import load_dotenv
import pandas as pd

# Cargar variables de entorno
load_dotenv()

# Crear directorio outputs si no existe
os.makedirs('outputs', exist_ok=True)

# Importar módulos personalizados
from longcontent_generator import core, scraper, utils, config

print("✅ Librerías importadas correctamente")
print(f"📊 Modelo Gemini: {config.CONFIG['gemini_model']}")
print(f"📁 Outputs en: {config.CONFIG['output_dir']}")

In [ ]:
# Cargar contexto de LEO (personalidad, proyecto, audiencia)
leo_context = core.load_leo_context()

print("\n📋 Contexto de LEO cargado:")
print(f"   • Personalidad: {len(leo_context['personality'])} caracteres")
print(f"   • Proyecto: {len(leo_context['project'])} caracteres")
print(f"   • Audiencia: {len(leo_context['audience'])} caracteres")

---

## 🎯 SELECCIONA TU MÉTODO DE CREACIÓN

Cambia el valor de `metodo_seleccionado` a una de estas opciones:
- `"topico"` - Generar a partir de un tema amplio
- `"keyword"` - Generar a partir de una keyword específica
- `"newsletter"` - Generar a partir de una URL de newsletter

**Ejecuta solo UNA de las secciones según tu elección.**

In [ ]:
# 🔧 CONFIGURA AQUÍ TU MÉTODO
metodo_seleccionado = "topico"  # Cambia a "topico", "keyword" o "newsletter"

print(f"🎯 Método seleccionado: {metodo_seleccionado.upper()}")

---

# 📝 MÉTODO 1: Generación por Tópico

Define un tema amplio y LEO sugerirá la mejor keyword para SEO.

In [ ]:
if metodo_seleccionado == "topico":
    # Define tu tópico aquí
    topico = "Inteligencia Artificial en la industria editorial y la creación de contenidos"
    
    print(f"📌 Tópico definido: {topico}")
    print("\n🤖 LEO está analizando el tópico y sugiriendo una keyword óptima...\n")
    
    # Sugerir keyword desde el tópico
    keyword_principal = core.suggest_keyword_from_topic(topico, leo_context)
    
    if keyword_principal:
        print(f"\n✨ Keyword sugerida: '{keyword_principal}'")
        print("\n💡 Puedes continuar con el pipeline usando esta keyword.")
    else:
        print("❌ No se pudo generar una keyword. Verifica la configuración de Gemini.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

---

# 🔑 MÉTODO 2: Generación por Keyword

Define directamente la keyword con la que quieres trabajar.

In [ ]:
if metodo_seleccionado == "keyword":
    # Define tu keyword aquí
    keyword_principal = "cómo usar IA para mejorar tu escritura creativa"
    
    print(f"🔑 Keyword definida: '{keyword_principal}'")
    print("\n💡 Continuando con el pipeline de investigación...")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

---

# 📰 MÉTODO 3: Generación desde Newsletter/URL

Analiza el contenido de una newsletter o artículo para generar ideas y contenido.

In [ ]:
if metodo_seleccionado == "newsletter":
    # Pega aquí la URL de "Ver en navegador" de la newsletter
    newsletter_url = "https://bbvm6.r.sp1-brevo.net/mk/mr/sh/1t6AVsd2XFnIGF8hGeIHTS2plaTU6I/v-MLKzmEd6I2"
    
    print(f"📰 URL de newsletter: {newsletter_url}")
    print("\n🔍 LEO está analizando el contenido...\n")
    
    # Extraer y analizar el contenido
    newsletter_analysis = core.extract_and_summarize_url(newsletter_url, leo_context)
    
    if newsletter_analysis:
        print("\n" + "="*70)
        print("📊 ANÁLISIS DEL NEWSLETTER")
        print("="*70)
        print(newsletter_analysis['raw_analysis'])
        print("="*70)
        
        # Guardar el análisis
        with open('outputs/analisis_newsletter.md', 'w', encoding='utf-8') as f:
            f.write(newsletter_analysis['raw_analysis'])
        
        print("\n💾 Análisis guardado en 'outputs/analisis_newsletter.md'")
        print("\n💡 Usa los 'Temas Sugeridos' para generar contenido.")
    else:
        print("❌ No se pudo analizar el contenido de la URL.")
else:
    print("⏭️  Método no seleccionado. Pasa a la siguiente sección.")

---

# 🔬 PASO 1: Investigación de Keywords (Opcional)

Si elegiste método 1 o 2, puedes hacer scraping de keywords relacionadas.

In [ ]:
# Solo ejecutar si NO estás usando el método newsletter
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔍 Scrapeando keywords relacionadas con: '{keyword_principal}'")
    
    # Configuración del scraper
    scraper_config = {
        'keyword': keyword_principal,
        'language': 'es',
        'country': 'es',
        'scrape_levels': 1,  # Nivel de profundidad
        'headless': True
    }
    
    # Crear instancia del scraper
    kw_scraper = scraper.GoogleKeywordScraper(**scraper_config)
    
    # Ejecutar scraping
    keywords_df = kw_scraper.run()
    
    if keywords_df is not None and not keywords_df.empty:
        # Guardar resultados
        keywords_df.to_csv('outputs/keywords_scraped.csv', index=False)
        print(f"\n✅ {len(keywords_df)} keywords encontradas y guardadas")
        print("\n📋 Primeras 10 keywords:")
        print(keywords_df.head(10))
    else:
        print("⚠️  No se encontraron keywords. Continuando sin ellas.")
else:
    print("⏭️  Saltando investigación de keywords (método newsletter).")

---

# 🌐 PASO 2: Búsqueda de Artículos de Referencia

Busca artículos relacionados para usar como contexto.

In [ ]:
if metodo_seleccionado in ["topico", "keyword"]:
    print(f"🔎 Buscando artículos sobre: '{keyword_principal}'")
    
    # Buscar en Google Custom Search
    search_results = core.google_custom_search(
        query=keyword_principal,
        country='ES',
        max_results=5
    )
    
    if not search_results.empty:
        search_results.to_csv('outputs/search_results.csv', index=False)
        print(f"\n✅ {len(search_results)} artículos encontrados")
        print("\n📰 Artículos encontrados:")
        for idx, row in search_results.iterrows():
            print(f"   {idx+1}. {row['title']}")
            print(f"      {row['link']}\n")
    else:
        print("⚠️  No se encontraron artículos.")
else:
    print("⏭️  Saltando búsqueda de artículos (método newsletter).")

---

# 📚 PASO 3: Scraping de Contenido de Artículos

In [ ]:
if metodo_seleccionado in ["topico", "keyword"]:
    if not search_results.empty:
        print("📖 Scrapeando contenido de los artículos...\n")
        
        scraped_content = []
        
        for idx, row in search_results.iterrows():
            url = row['link']
            print(f"   Scrapeando {idx+1}/{len(search_results)}: {row['title'][:60]}...")
            
            content = core.scrape_article(url)
            
            if content and len(content) > 100:
                scraped_content.append({
                    'title': row['title'],
                    'url': url,
                    'content': content[:3000]  # Limitar a 3000 caracteres
                })
        
        # Guardar contenido scrapeado
        scraped_df = pd.DataFrame(scraped_content)
        if not scraped_df.empty:
            scraped_df.to_csv('outputs/scraped_articles.csv', index=False)
            print(f"\n✅ {len(scraped_df)} artículos scrapeados exitosamente")
        else:
            print("⚠️  No se pudo scrapear ningún artículo")
    else:
        print("⚠️  No hay artículos para scrapear")
        scraped_content = []
else:
    print("⏭️  Usando contenido de newsletter como contexto.")
    scraped_content = []

---

# ✍️ PASO 4: Generación del Artículo Final

LEO generará un artículo completo usando toda la información recopilada.

In [ ]:
print("🎨 Preparando contexto para generación del artículo...\n")

# Preparar contexto según el método
context_sources = []

if metodo_seleccionado == "newsletter":
    # Usar el análisis del newsletter como contexto
    if newsletter_analysis:
        context_sources.append(newsletter_analysis['raw_analysis'])
        context_sources.append(newsletter_analysis['original_content'])
        
        # Definir keyword desde el análisis
        print("💡 Por favor, elige uno de los 'Temas Sugeridos' del análisis como keyword:")
        keyword_principal = input("Keyword para el artículo: ")
    else:
        print("❌ No hay análisis de newsletter disponible")
        keyword_principal = None
else:
    # Usar artículos scrapeados como contexto
    if scraped_content:
        for article in scraped_content:
            context_sources.append(f"# {article['title']}\n\n{article['content']}")
        print(f"✅ Usando {len(scraped_content)} artículos como contexto")
    else:
        print("⚠️  No hay artículos scrapeados. Generando solo con la keyword.")

# Generar artículo
if keyword_principal and context_sources:
    print(f"\n🚀 Generando artículo sobre: '{keyword_principal}'\n")
    
    articulo = core.generate_article_with_context(
        keyword=keyword_principal,
        context_sources=context_sources,
        leo_context=leo_context
    )
    
    if articulo:
        # Guardar artículo
        with open('outputs/articulo_leo_generado.md', 'w', encoding='utf-8') as f:
            f.write(articulo)
        
        print("\n" + "="*70)
        print("📝 ARTÍCULO GENERADO")
        print("="*70)
        print(articulo[:1000] + "\n...\n")
        print("="*70)
        print(f"\n💾 Artículo completo guardado en 'outputs/articulo_leo_generado.md'")
        print(f"📊 Longitud: {len(articulo)} caracteres, ~{len(articulo.split())} palabras")
    else:
        print("❌ Error al generar el artículo")
else:
    print("❌ Falta keyword o contexto para generar el artículo")

---

# 📊 Resumen Final

In [ ]:
print("="*70)
print("📊 RESUMEN DEL PROCESO")
print("="*70)
print(f"\n🎯 Método utilizado: {metodo_seleccionado.upper()}")
print(f"🔑 Keyword principal: {keyword_principal}")

if metodo_seleccionado in ["topico", "keyword"]:
    print(f"📚 Artículos scrapeados: {len(scraped_content) if scraped_content else 0}")
elif metodo_seleccionado == "newsletter":
    print(f"📰 Newsletter analizada: {'Sí' if newsletter_analysis else 'No'}")

print(f"\n✅ Archivos generados en outputs/:")
for file in ['articulo_leo_generado.md', 'analisis_newsletter.md', 'keywords_scraped.csv', 
             'search_results.csv', 'scraped_articles.csv']:
    filepath = f'outputs/{file}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"   • {file} ({size:,} bytes)")

print("\n" + "="*70)
print("🎉 ¡Proceso completado!")
print("="*70)